In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import lightgbm as lgb

SEED = 42
np.random.seed(SEED)

In [2]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])

In [3]:
def create_features(df, kmeans=None, scaler=None, fit=False):
    data = df.copy()

    data['total_bees'] = (
        data['honeybee'] +
        data['bumbles'] +
        data['andrena'] +
        data['osmia']
    )

    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)
    data['osmia_honeybee'] = data['osmia'] * data['honeybee']

    data['temp_range'] = data['MaxOfUpperTRange'] - data['MinOfLowerTRange']

    data['avg_temp'] = (
        data['AverageOfUpperTRange'] +
        data['AverageOfLowerTRange']
    ) / 2

    data['temp_x_rain'] = data['avg_temp'] * data['RainingDays']

    for col in ['clonesize', 'total_bees', 'fruitmass', 'seeds']:
        data[f'log_{col}'] = np.log1p(data[col])

    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)

    cluster_cols = ['clonesize', 'total_bees', 'avg_temp', 'RainingDays']

    if fit:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(data[cluster_cols])
        kmeans = KMeans(n_clusters=6, random_state=SEED, n_init=20)
        data['cluster'] = kmeans.fit_predict(scaled)
    else:
        scaled = scaler.transform(data[cluster_cols])
        data['cluster'] = kmeans.predict(scaled)

    return data, kmeans, scaler


train_fe, kmeans, scaler = create_features(train, fit=True)
test_fe, _, _ = create_features(test, kmeans=kmeans, scaler=scaler)

X = train_fe.drop(columns=['yield'])
y = train_fe['yield'].astype(np.float32)

In [4]:
params = {
    'objective': 'regression_l1',
    'metric': 'mae',

    'learning_rate': 0.028,
    'num_leaves': 80,
    'min_data_in_leaf': 22,

    'feature_fraction': 0.9,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,

    'lambda_l1': 0.3,
    'lambda_l2': 0.7,

    'boosting': 'gbdt',

    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,

    'verbosity': -1,
    'seed': SEED
}

# ===============================
# CV + WINSORIZED MAE
# ===============================
kf = KFold(n_splits=10, shuffle=True, random_state=SEED)

test_preds = np.zeros(len(test_fe))
maes = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f'Fold {fold}/10')

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    train_set = lgb.Dataset(X_tr, y_tr)
    val_set   = lgb.Dataset(X_val, y_val)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=5200,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(220, verbose=False)]
    )

    val_pred = model.predict(X_val)

    # 🔥 winsorize (very light)
    lo, hi = np.percentile(val_pred, [1, 99])
    val_pred = np.clip(val_pred, lo, hi)

    fold_mae = mean_absolute_error(y_val, val_pred)
    maes.append(fold_mae)

    test_preds += model.predict(test_fe) / kf.n_splits

print('\n🏁 FINAL OOF MAE:', np.mean(maes), '±', np.std(maes))


Fold 1/10


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Fold 2/10
Fold 3/10
Fold 4/10
Fold 5/10
Fold 6/10
Fold 7/10
Fold 8/10
Fold 9/10
Fold 10/10

🏁 FINAL OOF MAE: 248.31094425143996 ± 5.624094625536083


In [5]:
submission = pd.DataFrame({
    'id': test_ids,
    'yield': np.clip(test_preds, y.min(), y.max())
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,yield
0,15000,7507.239777
1,15001,5882.871566
2,15002,6463.239691
3,15003,4668.775869
4,15004,5920.689080
